<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c11-header.png" alt="Nextia Learning · Evaluating and Testing AI Systems" width="100%">

# Evaluate summaries against expert ratings

**[Evaluate summaries against expert ratings](https://learning.nextia-ai.com/courses/evals/m07/evaluate-summaries-against-expert-ratings/)** · Evaluating and Testing AI Systems · Module 7, case study 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** find out which automatic measures of summary quality agree with experts, and on which quality. You compare length, ROUGE, copied word pairs, embedding similarity and two model judges with the ratings of three summarisation experts, on 640 summaries that no choice was tuned on. You report rank correlations with intervals, read where each measure fails, and decide which checks to automate and which need a person.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in a few minutes (most of it is the download and the embeddings). |
| **Needs** | An internet connection for the setup cells (about 37 MB). No account, no key, no graphics card: every model judge score is a real recording. |
| **Recorded** | The model judges: gpt-6-luna as `chat-small` and gpt-6.1-sol as `chat-strong`, on Azure, 2026-10-09, on the 800 summaries of 50 articles. |
| **You should know** | [Generated-output metrics](https://learning.nextia-ai.com/courses/evals/m03/generated-output-metrics/), [Model-based judging](https://learning.nextia-ai.com/courses/evals/m04/model-based-judging/) and [Uncertainty and slices](https://learning.nextia-ai.com/courses/evals/m05/uncertainty-and-slices/) in this course. |
| **Data** | SummEval (Fabbri et al., 2021), MIT licence: 1,600 summaries of 100 CNN/DailyMail news articles by 16 systems, each rated by 3 experts and 5 crowd workers on 4 qualities. The articles are downloaded from their source. |
| **Tested** | Python 3.12 with pandas 3.0.6, scipy 1.18.1, rouge-score 0.1.2 and sentence-transformers 6.1.0 (torch 2.14.1, CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/evals/m07/evaluate-summaries-against-expert-ratings/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C11/M07-L02-evaluate-summaries-against-expert-ratings/evaluate-summaries-against-expert-ratings-solution.ipynb).

## Setup: packages

Run the next cell. It uses `pandas`, `scipy`, `matplotlib` and `sentence-transformers`, which Colab and Kaggle already have, and installs `rouge-score` 0.1.2 (the ROUGE implementation from Google Research). It installs any other package that is missing. You should see one line with the versions.

On your own computer, `sentence-transformers` also installs PyTorch (about 1 GB). The notebook needs no account and no key: every model judge score comes from a recording.

In [ ]:
import importlib, platform, subprocess, sys
for module, package in (("rouge_score", "rouge-score==0.1.2"), ("sentence_transformers", "sentence-transformers"),
                        ("scipy", "scipy"), ("pandas", "pandas"), ("pyarrow", "pyarrow"), ("matplotlib", "matplotlib")):
    try:
        importlib.import_module(module)
    except ImportError:  # not installed: install it
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
import pandas, scipy, sentence_transformers
print("Python", platform.python_version(), "| pandas", pandas.__version__, "| scipy", scipy.__version__,
      "| sentence-transformers", sentence_transformers.__version__)

## Setup: the data and the recorded judge scores

Run the next cell. It makes a folder `summary-eval` and downloads into it:

- the **SummEval** ratings and summaries (5.8 MB) from the authors' own link;
- the **CNN/DailyMail** test file (30 MB) from Hugging Face, at one fixed version, for the 100 articles and their original highlights;
- the course's recorded judge scores, the judge prompt and the new article (about 0.6 MB) from the labs repository.

It checks the checksum of every file. You should see `Ready: 1600 summaries of 100 articles, 16 systems, 2 judge recordings.`

If the cell shows an error, read its last line. `Could not download`: check your internet connection and run the cell again (in Kaggle, turn on **Internet** in the settings). `wrong checksum`: delete the folder `summary-eval` and run the cell again.

Ratings and summaries: SummEval (Fabbri et al., 2021), github.com/Yale-LILY/SummEval, MIT licence. Articles: the CNN/DailyMail dataset (Hermann et al., 2015; See et al., 2017); the texts belong to CNN and the Daily Mail, so the course does not copy them: you download them from the source. We changed nothing in the files.

In [ ]:
import hashlib, json, os, random, re, subprocess, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C11/M07-L02-evaluate-summaries-against-expert-ratings/")
FILES = {
    "data/summeval.jsonl": ("https://storage.googleapis.com/sfr-summarization-repo-research/model_annotations.aligned.jsonl", "f0d4166e0cdeb439b387c4449634b067b7e9f721b9ef4c2b722f6b7550d3d6ab"),
    "data/cnndm_test.parquet": ("https://huggingface.co/datasets/abisee/cnn_dailymail/resolve/96df5e686bee6baa90b8bee7c28b81fa3fa6223d/3.0.0/test-00000-of-00001.parquet", "04e322d2634a96dba76bf9a6294fbbe48e0b36abeae43f13d86ba2c3bebffe4e"),
    "new_input/article.json": (LABS + "new_input/article.json", "59cfc48f96349d53705d5ae51c76f40f6b6df2cfcfe6cb0f1b0527fee7d82bb9"),
    "new_input/judgements.json": (LABS + "new_input/judgements.json", "1abb2ef78c1d8684db928e5cd6ac99ac009cfa0c58bde6b0bb73505e698de54a"),
    "prompts/judge_v1.md": (LABS + "prompts/judge_v1.md", "8e70441e3fe1c48e1c7075e8a1ba37ea917e08a5572183ed670418d4c0393a3c"),
    "recordings/judge_chat-small.jsonl": (LABS + "recordings/judge_chat-small.jsonl", "e191e53652dac3dec391a654578f95ca87c9a7582534c5ed09c9d48e62cb9e5b"),
    "recordings/judge_chat-strong.jsonl": (LABS + "recordings/judge_chat-strong.jsonl", "f3980b4a80c143418900b585468499e7ee3dde2e55813fc1786df374c0f4e5f6"),
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
WORK = Path("summary-eval").resolve()
for folder in ("data", "recordings", "prompts", "new_input"):
    (WORK / folder).mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, (url, expected) in FILES.items():
    path = WORK / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(url, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder summary-eval and run again.")
os.chdir(WORK)

DIMS = ["coherence", "consistency", "fluency", "relevance"]
raw = [json.loads(line) for line in Path("data/summeval.jsonl").read_text(encoding="utf-8").splitlines()]
rows = []
for r in raw:
    article = r["id"].split("-test-")[1]              # SummEval ID = source + "-test-" + CNN/DailyMail ID
    row = {"id": r["id"], "article": article, "system": r["model_id"], "summary": r["decoded"]}
    for d in DIMS:
        row[f"expert_{d}"] = sum(a[d] for a in r["expert_annotations"]) / 3   # the mean of the 3 experts
        row[f"crowd_{d}"] = sum(a[d] for a in r["turker_annotations"]) / 5    # the mean of the 5 crowd workers
        row[f"experts_{d}"] = [a[d] for a in r["expert_annotations"]]         # the three expert ratings
    rows.append(row)
df = pd.DataFrame(rows).sort_values(["article", "system"]).reset_index(drop=True)

cnndm = pd.read_parquet("data/cnndm_test.parquet")
cnndm = cnndm[cnndm["id"].isin(set(df["article"]))]
articles = {r.id: r.article for r in cnndm.itertuples()}
highlights = {r.id: r.highlights for r in cnndm.itertuples()}

def load_judge(model):
    return [json.loads(line) for line in Path(f"recordings/judge_{model}.jsonl").read_text(encoding="utf-8").splitlines()]

judges = {m: load_judge(m) for m in ("chat-small", "chat-strong")}
print(f"Ready: {len(df)} summaries of {df['article'].nunique()} articles, {df['system'].nunique()} systems, "
      f"{len(judges)} judge recordings.")

## Setup: helpers

The next cell defines the check helpers and the two tools of this case study:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `spearman(x, y)`: Spearman's rank correlation. Section 3 explains it.
- `with_interval(data, measure, quality)`: the correlation of a measure with the experts' mean, with a 95% bootstrap interval. It resamples **whole articles**, 2,000 times, as in [Uncertainty and slices](https://learning.nextia-ai.com/courses/evals/m05/uncertainty-and-slices/).

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

from scipy.stats import rankdata

def spearman(x, y):
    """Spearman's rank correlation: the ordinary correlation of the two lists of ranks (ties share a rank)."""
    rx, ry = rankdata(x), rankdata(y)
    if np.std(rx) == 0 or np.std(ry) == 0:
        return float("nan")
    return float(np.corrcoef(rx, ry)[0, 1])

def with_interval(data, measure, quality, resamples=2000, seed=2026):
    """Spearman of `measure` with the experts' mean on `quality`, and a 95% interval that resamples whole articles."""
    data = data[data[measure].notna()]
    x, y = data[measure].to_numpy(float), data[f"expert_{quality}"].to_numpy(float)
    names = sorted(data["article"].unique())
    rows_of = {a: np.flatnonzero(data["article"].to_numpy() == a) for a in names}
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(resamples):
        pick = np.concatenate([rows_of[names[j]] for j in rng.integers(0, len(names), len(names))])
        boots.append(spearman(x[pick], y[pick]))
    low, high = np.nanpercentile(boots, [2.5, 97.5])
    return round(spearman(x, y), 4), round(float(low), 4), round(float(high), 4)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data, and who rated it

One row is **one summary**. The experts rated four qualities from 1 (worst) to 5 (best): **coherence** (the summary as a whole is well organised), **consistency** (every fact is supported by the article), **fluency** (each sentence is well formed) and **relevance** (it keeps what matters, and nothing else).

The SummEval paper says who rated. The **experts** were three people "who have written papers on summarization either for academic conferences (2) or as part of a senior thesis (1)". They rated in two rounds, and in the second round they could see each other's first scores. The **crowd** ratings come from five Amazon Mechanical Turk workers per summary.

Run the next cell. It shows the 16 summaries of one article (an auction of a dress from the film *Gone With the Wind*), with the experts' mean ratings. You should see the summaries sorted from the best to the worst relevance.

In [ ]:
DRESS = "20422c3f"   # the last characters of the article's ID
one = df[df["article"].str.endswith(DRESS)].sort_values("expert_relevance", ascending=False)
pd.set_option("display.max_colwidth", 90)
one[["system", "summary"] + [f"expert_{d}" for d in DIMS]].round(2)

> **Check.** Most summaries are in lower case, with spaces around punctuation: they are the systems' raw output. The top one, from system M0, is the article's first three sentences. Some abstractive systems write errors: M10 says that "scarlett o'hara sold for $ 137,000".

Next, the shape of the ratings. Run the cell. It counts the summaries by their expert mean, rounded, per quality.

In [ ]:
pd.DataFrame({d: df[f"expert_{d}"].round().value_counts().sort_index() for d in DIMS}).fillna(0).astype(int)

**What this tells us.** Coherence and relevance spread over 2 to 5. Consistency and fluency do not: 1,387 of 1,600 summaries round to 5 on consistency, and 1,291 on fluency. Few summaries have a factual error, so any correlation on consistency rests on those few.

## 2. Split the articles, then measure the ceiling

The 16 summaries of one article share its topic and its difficulty, so the unit of the split is the **article**. The case study uses the course's habit: 10 **dev** articles, where the judge prompt was tried, and 40 **holdout** articles, where every measure is compared once. The other 50 articles were not judged, to keep the recording within its budget.

Run the next cell. You should see `dev 160 | holdout 640 | unjudged 800`.

In [ ]:
order = sorted(df["article"].unique())
random.Random(2026).shuffle(order)
split_of = {a: ("dev" if i < 10 else "holdout" if i < 50 else "unjudged") for i, a in enumerate(order)}
df["split"] = df["article"].map(split_of)
holdout = df[df["split"] == "holdout"]
print(" | ".join(f"{k} {v}" for k, v in df["split"].value_counts().sort_index().items()))

How far can a measure agree with the experts? Not further than the experts agree with each other. The next cell compares each expert with the mean of the other two, on the holdout, and averages the three results. That is the **ceiling**.

It also compares the **crowd's** mean with the experts' mean.

> **Predict.** Five crowd workers per summary, against three experts. Will the crowd's mean agree with the experts as well as the experts agree with each other? Write your guess, then run the cell.

In [ ]:
for d in DIMS:
    ceiling = np.mean([spearman([e[i] for e in holdout[f"experts_{d}"]],
                                [(sum(e) - e[i]) / 2 for e in holdout[f"experts_{d}"]]) for i in range(3)])
    crowd = spearman(holdout[f"crowd_{d}"], holdout[f"expert_{d}"])
    print(f"{d:12} one expert vs the other two {ceiling:+.2f} | crowd vs experts {crowd:+.2f}")

**What this tells us.** The experts agree well with each other (between 0.55 and 0.75). The crowd's mean hardly agrees with them at all (at most 0.12). The SummEval paper found the same. Who rates matters: a human rating is not good evidence by itself. The experts' second round makes their agreement a little optimistic, because they saw each other's scores.

## 3. Rank correlation on five summaries

The experts give scores from 1 to 5, ROUGE gives numbers between 0 and 1, a judge gives whole numbers. The scales differ, so we compare **orders**: does the measure put the summaries in the same order as the experts? **Spearman's rank correlation** does this. Rank both lists, then compute the ordinary correlation of the ranks. With no ties, it equals 1 − 6 × Σd² / (n(n² − 1)), where d is the difference of the two ranks of one summary. It is +1 for the same order, 0 for no relation, −1 for the reverse order.

Here are five of the dress article's summaries: the experts' mean relevance, and ROUGE-2, which section 5 explains.

> **Your turn.** Rank both lists (1 = best), compute d for each summary, and Spearman with the formula. Then compare with `spearman()`.

In [ ]:
expert = {"A": 4.00, "B": 3.00, "C": 3.67, "D": 3.33, "E": 4.67}
rouge2 = {"A": 0.257, "B": 0.265, "C": 0.305, "D": 0.179, "E": 0.308}

d = {...}            # Your code here: rank difference per summary, experts' rank minus ROUGE-2's rank
rho_by_hand = ...    # Your code here: 1 - 6 * (sum of d squared) / (n * (n * n - 1))
print(d, rho_by_hand, spearman(list(expert.values()), list(rouge2.values())))

In [ ]:
expect('Spearman by hand', rho_by_hand, 0.5)

> **Check.** Summary E is first in both orders. A, B, C and D swap places, so ρ = 0.5: the orders agree partly. The [lesson page](https://learning.nextia-ai.com/courses/evals/m07/evaluate-summaries-against-expert-ratings/#rank-correlation-do-two-orders-agree) has the picture.

## 4. The baseline: length

The cheapest measure of all is the length of the summary in words. Every other measure must beat it, or it adds nothing.

Run the next cell. It computes the length, and its correlation with the experts on the holdout, with a 95% interval. It takes a few seconds.

In [ ]:
df["length"] = df["summary"].str.split().str.len()
holdout = df[df["split"] == "holdout"]
for q in DIMS:
    print(f"{q:12}", with_interval(holdout, "length", q))

**What this tells us.** Length knows nothing about quality, and on three qualities its correlation is near 0. On **relevance** it reaches +0.29 (+0.19 to +0.38): longer summaries keep more of the article, and the experts noticed. Any measure that claims to see relevance must beat this.

## 5. ROUGE: word overlap with a reference

**ROUGE** counts the words (ROUGE-1) or the word pairs (ROUGE-2) that the summary shares with a **reference** summary written by a person. ROUGE-L uses the longest sequence of words that both share in the same order. Each is an F1 score of the shared part, between 0 and 1. Here the reference is the article's original highlights.

Run the next cell. It computes the three ROUGE scores for all 1,600 summaries with `rouge-score`, with the stemmer on (so "sold" and "sells" match).

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
scores = [scorer.score(highlights[a], s) for a, s in zip(df["article"], df["summary"])]
for name in ("rouge1", "rouge2", "rougeL"):
    df[name] = [round(x[name].fmeasure, 4) for x in scores]
df[df["article"].str.endswith(DRESS)][["system", "rouge1", "rouge2", "rougeL", "expert_relevance"]].round(3)

> **Check.** For the dress article you should see ROUGE-2 0.257 for M0 and 0.308 for M13, as in section 3.

## 6. Two more free measures: copied word pairs and embedding similarity

**Copied word pairs** is the share of the summary's word pairs that also appear in the article. It needs no reference. A summary that copies the article cannot easily invent a fact, so it may track consistency.

Run the next cell. It lower-cases both texts, keeps letters and digits as words, and compares the consecutive word pairs. You should see M0, which copies the article's first sentences, at 1.0.

In [ ]:
def words(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def copied_pairs(summary, article):
    """The share of the summary's consecutive word pairs that are also in the article."""
    s, a = words(summary), words(article)
    pairs = list(zip(s, s[1:]))
    seen = set(zip(a, a[1:]))
    return round(sum(p in seen for p in pairs) / len(pairs), 4) if pairs else 1.0

df["copied"] = [copied_pairs(s, articles[a]) for s, a in zip(df["summary"], df["article"])]
holdout = df[df["split"] == "holdout"]
df[df["article"].str.endswith(DRESS)][["system", "copied", "expert_consistency"]].round(3)

> **Your turn.** Which system copies the least? Find the system with the lowest **mean** share of copied word pairs on the holdout, and store its name (for example `"M5"`) in `least_copying`. Then look at its experts' mean consistency.

In [ ]:
least_copying = ...   # Your code here: group the holdout by system, take the mean of "copied", find the lowest
least_copying

In [ ]:
expect_hash('the system that copies least', least_copying, 'b3abac1f72b58308')

> **Check.** The system that copies least is also one of the two with the most factual errors: its experts' mean consistency is far below the others. Writing in your own words is where facts go wrong.

**Embedding similarity** turns each text into a vector with a small sentence-embedding model (`all-MiniLM-L6-v2`, 22 million parameters, Apache-2.0) and takes the cosine similarity. It rewards the same meaning in other words, which ROUGE cannot. The model reads at most 256 word pieces, so it sees only the start of an article.

Run the next cell. It downloads the model (about 90 MB) and embeds 1,800 texts on the CPU. It takes about 10 to 60 seconds.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
                            device="cpu")
keys = sorted(articles)
art_vec = dict(zip(keys, embedder.encode([articles[k] for k in keys], normalize_embeddings=True)))
ref_vec = dict(zip(keys, embedder.encode([highlights[k] for k in keys], normalize_embeddings=True)))
sum_vec = embedder.encode(df["summary"].tolist(), normalize_embeddings=True, batch_size=64)
df["emb_reference"] = [round(float(v @ ref_vec[a]), 4) for v, a in zip(sum_vec, df["article"])]
df["emb_article"] = [round(float(v @ art_vec[a]), 4) for v, a in zip(sum_vec, df["article"])]
print(df[["emb_reference", "emb_article"]].describe().loc[["mean", "min", "max"]].round(3))

## 7. The model judges

A **model judge** reads the article and the summary and rates the four qualities with a rubric, as in [Model-based judging](https://learning.nextia-ai.com/courses/evals/m04/model-based-judging/). The rubric gives each quality its own definition and anchors for 1, 3 and 5, written from the SummEval definitions. One call rates all four qualities and copies the first unsupported statement, if any.

Run the next cell. It prints the prompt, exactly as it was sent.

In [ ]:
print(Path("prompts/judge_v1.md").read_text())

The course recorded both judges on the 50 judged articles: gpt-6-luna as `chat-small` and gpt-6.1-sol as `chat-strong`, on Azure, on 2026-10-09. The prompt was written once and tried on the 10 dev articles; it did not change before the holdout run.

Run the next cell. It adds the judges' scores to the table and counts the summaries that have no score.

In [ ]:
for model, rows in judges.items():
    scored = {(r["id"], r["system"]): r["answer"] for r in rows if r["answer"]}
    for q in DIMS:
        df[f"{model}_{q}"] = [scored[(i, s)][q] if (i, s) in scored else None for i, s in zip(df["id"], df["system"])]
    refused = sorted({r["id"][-8:] for r in rows if r.get("error")})
    print(f"{model:12} {len(scored)} scored | refused by the content filter: {sum(1 for r in rows if r.get('error'))} "
          f"summaries of {len(refused)} articles {refused}")
holdout = df[df["split"] == "holdout"]

**What this tells us.** Azure's content filter refused every summary of three articles, on both deployments, and `chat-small` one more summary of a fourth. The prompt includes the article, and these articles report crimes or an injury. The judge cannot score these summaries at all. They need a person, whatever the judge's quality elsewhere.

## 8. The fair comparison: every measure, the same 640 summaries

Now every measure gets the same test: the holdout summaries, one metric (Spearman with the experts' mean), one interval method. Run the next cell. It takes about a minute: 10 measures × 4 qualities × 2,000 resamples.

In [ ]:
MEASURES = {"length": "Length", "rouge1": "ROUGE-1", "rouge2": "ROUGE-2", "rougeL": "ROUGE-L",
            "copied": "Copied word pairs", "emb_reference": "Embedding vs reference", "emb_article": "Embedding vs article"}
results = {}
for q in DIMS:
    for col, label in list(MEASURES.items()) + [(f"crowd_{q}", "Crowd (mean of 5)"), (f"chat-small_{q}", "Judge chat-small"),
                                                (f"chat-strong_{q}", "Judge chat-strong")]:
        results[(label, q)] = with_interval(holdout, col, q)
table = pd.DataFrame({q: {label: f"{v[0]:+.2f} [{v[1]:+.2f}, {v[2]:+.2f}]" for (label, qq), v in results.items() if qq == q}
                      for q in DIMS})
table

**What this tells us.** Read it by column: each quality has its own winner.

- **Coherence:** the judges (+0.57 and +0.65) are far ahead of every free measure (copied word pairs +0.22, ROUGE-L +0.21).
- **Consistency:** the judges again (+0.60, +0.63). The best free measure is copied word pairs (+0.36); ROUGE sees little (+0.17).
- **Fluency:** the free measures are weak (at most +0.23); the judges reach +0.59 and +0.62, as far as the experts' ceiling.
- **Relevance:** the judges (+0.49, +0.54) beat ROUGE (+0.36 for ROUGE-1), and ROUGE barely beats length (+0.29).

The next cell draws the same results as a chart, with the experts' ceiling as a dashed line.

In [ ]:
import matplotlib.pyplot as plt

labels = list(dict.fromkeys(label for label, _ in results))
fig, axes = plt.subplots(1, 4, figsize=(15, 4.8), sharey=True)
for ax, q in zip(axes, DIMS):
    for i, label in enumerate(labels):
        v, lo, hi = results[(label, q)]
        ax.errorbar(v, i, xerr=[[v - lo], [hi - v]], fmt="o", color="#B4461C" if "Judge" in label else "#0054D8")
    ceiling = np.mean([spearman([e[i] for e in holdout[f"experts_{q}"]], [(sum(e) - e[i]) / 2 for e in holdout[f"experts_{q}"]])
                       for i in range(3)])
    ax.axvline(ceiling, ls="--", color="grey")
    ax.axvline(0, color="black", lw=0.5)
    ax.set_title(q)
    ax.set_xlim(-0.3, 1)
axes[0].set_yticks(range(len(labels)), labels)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Where each measure fails

A correlation is a summary of many cases. Read the cases behind it, as in [Generated-output metrics](https://learning.nextia-ai.com/courses/evals/m03/generated-output-metrics/#read-every-failure-before-you-count-it).

Some articles report crimes. The next cells show only the articles on neutral topics (sport, entertainment, animals), listed in `NEUTRAL`.

**ROUGE rewards copying the reference's words, not good summaries.** Run the next cell. It shows, for the holdout, the summaries that ROUGE-2 ranks far above the experts on relevance, and far below.

In [ ]:
NEUTRAL = ['20422c3f', '0054b8ce', '11c4ba9f', '139c9415', '21fe5d20', '2db8ec5c', '3205d8f4', '4c81e857', '4d90dea1', '52238ea6', '5335d96a', '54eb15df', '71da96cf', '7215a78b', '74c4b3c0', '787d8d7b', '7f10217b', '99e83812', 'a29453ce', 'b400eee2', 'b58bef19', 'bc62470d', 'c212835b', 'c62b9926', 'ca253c08', 'cc4d834f', 'e63799d0', 'f8282cd2', 'fee2e34b']
h = holdout.copy()
h["rank_experts"] = h["expert_relevance"].rank(pct=True)
h["rank_rouge2"] = h["rouge2"].rank(pct=True)
h["gap"] = h["rank_rouge2"] - h["rank_experts"]
h = h[h["article"].str[-8:].isin(NEUTRAL)]   # rank among all 640, then show the neutral ones
cols = ["system", "summary", "rouge2", "expert_relevance"]
print("ROUGE-2 much higher than the experts:")
display(h.sort_values("gap", ascending=False)[cols].head(3).round(2))
print("ROUGE-2 much lower than the experts:")
display(h.sort_values("gap")[cols].head(3).round(2))

**The judge misses a wrong fact, or invents one.** Run the next cell. It lists holdout summaries where the strong judge and the experts disagree by 2 points or more on consistency, with the judge's own note on the first unsupported statement.

In [ ]:
notes = {(r["id"], r["system"]): r["answer"]["unsupported"] for r in judges["chat-strong"] if r["answer"]}
h["note"] = [notes.get((i, s), "") for i, s in zip(h["id"], h["system"])]
h["consistency_gap"] = h["chat-strong_consistency"] - h["expert_consistency"]
far = h[h["consistency_gap"].abs() >= 2].sort_values("consistency_gap")
print(len(far), "summaries with a gap of 2 or more")
far[["system", "summary", "expert_consistency", "chat-strong_consistency", "note"]].round(2)

Read the rows. In some, the judge flags a statement that the article does support in other words; in others, the judge accepts a summary that the experts marked down. A judge with a high correlation can still be wrong on the summary in front of you.

## 10. Change one thing: rank the systems, not the summaries

A team that chooses between **systems** compares their means, not single summaries. Run the next cell. It computes, for each measure, the mean per system (16 systems) on the holdout, and the Spearman of those 16 means with the experts' means.

> **Predict.** Will the correlations go up or down when you compare 16 means instead of 640 summaries?

In [ ]:
means = holdout.groupby("system").mean(numeric_only=True).round(6)   # rounded: equal means stay tied
level = {label: {q: round(spearman(means[col if col in means else f"{col}_{q}"], means[f"expert_{q}"]), 2) for q in DIMS}
         for col, label in list(MEASURES.items()) + [("chat-small", "Judge chat-small"), ("chat-strong", "Judge chat-strong")]}
pd.DataFrame(level).T

**What this tells us.** Averages remove much of the noise of single summaries, so most correlations go up: for example ROUGE-1 on relevance goes from +0.36 to +0.70. But 16 systems is a tiny sample: one system that moves changes the result a lot. A measure can rank systems well and still be useless for checking one summary. Say which level you report.

## 11. Failure case: the interval that is too narrow

The 16 summaries of one article are not 16 independent cases. They share the article's topic and difficulty. If you resample **summaries** instead of articles, the interval looks more certain than the data allows.

Run the next cell. It compares the two intervals for ROUGE-2 on relevance.

In [ ]:
def interval_by_summary(data, measure, quality, resamples=2000, seed=2026):
    x, y = data[measure].to_numpy(float), data[f"expert_{quality}"].to_numpy(float)
    rng = np.random.default_rng(seed)
    boots = [spearman(x[i], y[i]) for i in (rng.integers(0, len(x), len(x)) for _ in range(resamples))]
    low, high = np.percentile(boots, [2.5, 97.5])
    return round(float(low), 3), round(float(high), 3)

print("by article:", with_interval(holdout, "rouge2", "relevance")[1:])
print("by summary:", interval_by_summary(holdout, "rouge2", "relevance"), "<- too narrow")

> **Warning.** Resample the unit that was sampled. Here 40 articles were drawn, not 640 independent summaries. The same mistake hides in any evaluation set with several outputs per input.

## 12. A new article, end to end

Kestrel's question (the news-digest team of the lesson page) is about **new** summaries, which have no expert rating and no reference. The new input is a Wikinews article (CC BY 4.0) about paid parking at Sydney beaches. `chat-small` wrote a three-sentence summary of it (recorded). The course then **constructed** two faulty versions: one with a changed number, one that keeps side details and misses the main point.

Run the next cell. It computes the free measures that need no reference, and shows both judges' recorded verdicts.

In [ ]:
new_article = json.loads(Path("new_input/article.json").read_text())
new = json.loads(Path("new_input/judgements.json").read_text())
print(new_article["attribution"], "\n")
v_article = embedder.encode([new_article["article"]], normalize_embeddings=True)[0]
for name, s in new["summaries"].items():
    verdicts = {j["model"]: j["answer"] for j in new["judgements"] if j["summary"] == name}
    vec = embedder.encode([s["text"]], normalize_embeddings=True)[0]
    print(f"{name}{' (CONSTRUCTED)' if s['constructed'] else ''}: {s['text']}")
    print(f"  length {len(s['text'].split())} | copied word pairs {copied_pairs(s['text'], new_article['article']):.2f} "
          f"| embedding vs article {float(vec @ v_article):.2f}")
    for m, a in verdicts.items():
        print(f"  {m:12} " + " ".join(f"{q} {a[q]}" for q in DIMS) + (f" | unsupported: {a['unsupported']}" if a["unsupported"] else ""))
    print()

> **Check.** Both judges catch the changed number (consistency 3, with the sentence copied as unsupported) and the missed main point (relevance 1). The free measures barely move: one changed number leaves almost every word pair copied, and the side details are copied word for word. ROUGE cannot run at all: there is no reference.

## 13. Your turn: the recommendation

The lesson page's recommendation automates a check only where a measure's **lower** interval bound is clearly above zero, and keeps a person where the measure is far from the experts' ceiling.

> **Your turn.** For each quality, find the measure in `results` with the highest **lower** bound, among the free measures and the two judges (not the crowd). Store it as a dictionary `best = {"coherence": "...", ...}` with the labels used in `results`.

In [ ]:
best = ...   # Your code here
best

In [ ]:
expect_hash('the best measure for each quality', best, 'a95d80e1a21a1ac2')

### Optional: a live judge call

The cell below sends one summary to a live model through the OpenAI-compatible Chat Completions API: a free local model with [Ollama](https://ollama.com) (`http://localhost:11434/v1`, no key) or your own provider and key. It is **off** by default. To try it, set `LIVE = True` and the settings. One call on `chat-strong` cost about US$0.0032 at the prices of 2026-10-08. Never type a key into a notebook that you share: read it from an environment variable. A small local model is not calibrated against the experts: measure it as in section 8 before you trust it.

In [ ]:
LIVE = False
BASE_URL = os.environ.get("JUDGE_BASE_URL", "http://localhost:11434/v1")
MODEL = os.environ.get("JUDGE_MODEL", "gemma3:4b")
API_KEY = os.environ.get("JUDGE_API_KEY", "not-needed")

def build_request(article, summary, model_name):
    system, user = Path("prompts/judge_v1.md").read_text().split("[system]\n", 1)[1].split("[user]\n", 1)
    return {"model": model_name, "max_completion_tokens": 2000,
            "messages": [{"role": "system", "content": system.strip()},
                         {"role": "user", "content": user.strip().replace("{article}", article).replace("{summary}", summary)}]}

if LIVE:
    %pip install -q "openai>=1"
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY, max_retries=0, timeout=120)
    s = new["summaries"]["wrong_number"]["text"]
    response = client.chat.completions.create(**build_request(new_article["article"], s, MODEL))
    print(response.choices[0].message.content)
else:
    print("Live call skipped. Set LIVE = True to try one.")

## Recap

- Compare a measure with experts by **order**, per quality: Spearman's rank correlation, with an interval that resamples **articles**.
- On these 640 summaries, **ROUGE** agrees weakly with the experts on every quality, and on relevance it barely beats the summary's **length**. Copied word pairs is the best free signal for consistency.
- The **model judges** with a rubric per quality agree far better, close to the experts' own agreement on coherence and consistency, but they still miss and invent faults on single summaries, and the content filter refused whole articles.
- The **crowd's** ratings hardly agree with the experts: who rates matters.
- Automate a screen where a measure clearly agrees; keep a person for the rest, and measure your own judge on your own expert ratings before you trust it.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Regression-test a classifier release by slice](https://learning.nextia-ai.com/courses/evals/m07/regression-test-a-classifier-release/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/evals/m07/evaluate-summaries-against-expert-ratings/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). SummEval is MIT (Fabbri et al., 2021); the CNN/DailyMail articles belong to their publishers and are downloaded from the source. The new article is from Wikinews (CC BY 4.0); two of its summaries are constructed. The model judge scores are real recordings.